# 02 — Full Pipeline

Chain **cleaning → normalization → encoding → export** using the `Pipeline` API.

```bash
pip install quprep
```

In [ ]:
import numpy as np
import pandas as pd

from quprep import Pipeline
from quprep.clean.imputer import Imputer
from quprep.encode.angle import AngleEncoder
from quprep.export.qasm_export import QASMExporter

## 1. Messy dataset

Real-world data has missing values, outliers, and mixed scales.

In [ ]:
rng = np.random.default_rng(0)

df = pd.DataFrame(
    {
        "a": [1.2, None, 3.4, 2.1, 999.0, 2.8, None, 1.9],   # outlier + NaN
        "b": [0.5, 0.6, None, 0.4, 0.7, None, 0.3, 0.8],     # NaN
        "c": rng.uniform(1, 5, 8).tolist(),
        "d": rng.uniform(0, 1, 8).tolist(),
    }
)

df

## 2. Build and run the pipeline

The pipeline applies each stage in order. Normalization is selected automatically based on the encoder.

In [ ]:
pipeline = Pipeline(
    cleaner=Imputer(strategy="mean"),     # fill NaNs with column mean
    encoder=AngleEncoder(rotation="ry"),  # Ry rotation gates, auto-normalizes to [0, π]
    exporter=QASMExporter(),              # OpenQASM 3.0 output
)

result = pipeline.fit_transform(df)

print(f"Encoded {len(result.encoded)} samples")
print(f"Qubits per circuit : {result.encoded[0].metadata['n_qubits']}")
print(f"Circuit depth      : {result.encoded[0].metadata['depth']}")

## 3. Inspect the first circuit

In [ ]:
print(result.circuit)

## 4. All rotation angles

In [ ]:
import pandas as pd

angles_df = pd.DataFrame(
    [enc.parameters for enc in result.encoded],
    columns=[f"q{i}" for i in range(result.encoded[0].metadata["n_qubits"])],
)
angles_df

## 5. Save all circuits to disk (save_batch)

In [ ]:
exporter = QASMExporter()
paths = exporter.save_batch(result.encoded, "/tmp/quprep_circuits/", stem="circuit")
print(f"Saved {len(paths)} circuits → {paths[0].parent}/")
for p in paths:
    print(f"  {p.name}")

## 6. Pipeline without exporter

Omit the exporter to get `EncodedResult` objects directly.

In [ ]:
pipeline_no_export = Pipeline(
    cleaner=Imputer(strategy="median"),
    encoder=AngleEncoder(rotation="rz"),
)

result2 = pipeline_no_export.fit_transform(df)

print(type(result2.encoded[0]))
print(result2.encoded[0].parameters)
print(result2.encoded[0].metadata)

## 7. Save and reload a fitted pipeline

In [ ]:
import quprep as qd

pipeline.save("/tmp/quprep_pipeline.pkl")
loaded = qd.Pipeline.load("/tmp/quprep_pipeline.pkl")

X_new = df.fillna(df.mean(numeric_only=True)).to_numpy()[:3]
result3 = loaded.transform(X_new)
print(f"Loaded pipeline → {len(result3.encoded)} samples encoded")